# 00 - Informe (Secciones 1 y 2)

**Trabajo Práctico 1: Arquitectura e Ingesta en Capa Bronze**
Asignatura: Arquitectura y Gestión de Datos Masivos — Ponderación: 15% (Evaluación de la Era 1)

Este notebook documenta las **Secciones 1 y 2** de la entrega: el contexto de negocio, la ficha técnica de cada fuente y el diagrama de arquitectura inicial.

* **Sección 3** (Pipeline de Ingesta y Carga Incremental — código PySpark): notebooks [`02_Ingesta_Bronze_RUES.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/3744392461485811?o=7474658561793168), [`03_Ingesta_Bronze_TRM.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/3744392461485812?o=7474658561793168), [`04_Ingesta_Bronze_CIIU.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/3744392461485813?o=7474658561793168) y [`06_Ingesta_Bronze_SECOP.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/2279170452559878?o=7474658561793168) (uno por cada fuente).
* **Sección 4** (Validaciones Iniciales y Detección de Anomalías): notebook [`05_Validaciones_Bronze.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/1875149303955997?o=7474658561793168).

**Equipo de trabajo**:

* Jhoan Stiward Cordoba

* Juan Pablo Quintero

* Jaime Usuga

* Juan Camilo Herrera Osorio

---

# Sección 1: Contexto de Negocio y Resumen de Datos

## Justificación del dominio

El caso de estudio analiza la **salud del tejido empresarial colombiano en su contexto macroeconómico**, combinando cuatro fuentes complementarias:

1. **RUES (Registro Mercantil)**: describe el estado del universo empresarial del país — matrículas, renovaciones, cancelaciones y naturaleza jurídica de personas naturales, jurídicas y ESAL registradas ante las Cámaras de Comercio.
2. **TRM (Tasa Representativa del Mercado)**: es el indicador macroeconómico que más directamente afecta a las empresas con actividad de comercio exterior (importación/exportación, deuda en dólares, insumos importados).
3. **Catálogo CIIU (actividades económicas)**: fuente de referencia que traduce los códigos numéricos de actividad económica de RUES (`cod_ciiu_act_econ_pri`, etc.) a su descripción textual — sin ella, RUES por sí solo no permite saber a qué sector pertenece cada empresa.
4. **SECOP II (Contratos Públicos)**: describe la participación de empresas en contratación pública estatal — montos, modalidades, entidades contratantes y proveedores adjudicados. Permite cruzar el universo empresarial de RUES con su actividad contractual real en el Estado.

Las cuatro fuentes comparten el mismo dominio de negocio (**actividad económica y empresarial de Colombia**) y se complementan porque permiten, en capas posteriores (Silver/Gold), cruzar la dinámica de creación/cierre de empresas **por sector económico** (uniendo RUES con el catálogo CIIU) con el comportamiento cambiario del mismo periodo (TRM) y la participación contractual en el Estado (SECOP) — por ejemplo, para analizar si la volatilidad del dólar coincide con picos de cancelación de matrículas en sectores exportadores/importadores, o si las empresas más activas en SECOP son también las más estables en el registro mercantil.

## Ficha técnica — Fuente 1: RUES (Registro Mercantil)

| Campo | Detalle |
|---|---|
| **Nombre del dataset** | Personas Naturales, Personas Jurídicas y Entidades Sin Ánimo de Lucro (RUES) |
| **Origen** | Portal de Datos Abiertos de Colombia (datos.gov.co) — API REST Socrata (SoQL) |
| **Ficha del dataset** | https://www.datos.gov.co/Comercio-Industria-y-Turismo/Personas-Naturales-Personas-Jur-dicas-y-Entidades-/c82u-588k/about_data |
| **API (recurso)** | `https://www.datos.gov.co/resource/c82u-588k.json` |
| **Método de extracción** | HTTP GET paginado (`$limit` / `$offset`), formato JSON |
| **Volumen exacto** | 9.407.309 registros totales (verificado con `$select=count(*)` el 2026-09-14) |
| **Tamaño aproximado** | 36 columnas de texto × ~9.4M filas → varios GB en crudo (JSON) |
| **Recencia** | Actualización continua; el pipeline filtra por `fecha_actualizacion` |
| **Carga usada en el pipeline** | Modo mensual (`replaceWhere`) o modo diario incremental (watermark sobre `fecha_actualizacion`), ver `02_Ingesta_Bronze_RUES.ipynb` |
| **Tabla Bronze** | `Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api` |

### Diccionario de datos inicial (RUES)

Todas las columnas llegan como `text` desde la API:

| Columna | Tipo original | Descripción |
|---|---|---|
| codigo_camara | text | Código de la Cámara de Comercio administradora |
| camara_comercio | text | Nombre de la Cámara de Comercio |
| matricula | text | Número de matrícula mercantil |
| inscripcion_proponente | text | Número de inscripción como proponente (contratación pública) |
| razon_social | text | Razón social (personas jurídicas) |
| primer_apellido | text | Primer apellido (personas naturales) |
| segundo_apellido | text | Segundo apellido (personas naturales) |
| primer_nombre | text | Primer nombre (personas naturales) |
| segundo_nombre | text | Segundo nombre (personas naturales) |
| sigla | text | Sigla o nombre abreviado |
| codigo_clase_identificacion | text | Código del tipo de documento de identificación |
| clase_identificacion | text | Tipo de documento (CC, NIT, CE, etc.) |
| numero_identificacion | text | Número de identificación |
| nit | text | Número de Identificación Tributaria |
| digito_verificacion | text | Dígito de verificación del NIT |
| cod_ciiu_act_econ_pri | text | Código CIIU de la actividad económica principal |
| cod_ciiu_act_econ_sec | text | Código CIIU de la actividad económica secundaria |
| ciiu3 | text | Código CIIU adicional (tercera actividad) |
| ciiu4 | text | Código CIIU adicional (cuarta actividad) |
| fecha_matricula | text (`YYYYMMDD`) | Fecha de matrícula inicial |
| fecha_renovacion | text (`YYYYMMDD`) | Fecha de la última renovación |
| ultimo_ano_renovado | text | Último año renovado |
| fecha_vigencia | text (`YYYYMMDD`) | Fecha de vigencia de la matrícula (usa `99991231` como valor centinela "sin vencimiento") |
| fecha_cancelacion | text (`YYYYMMDD`) | Fecha de cancelación de la matrícula (si aplica) |
| codigo_tipo_sociedad | text | Código del tipo societario |
| tipo_sociedad | text | Tipo de sociedad (S.A.S., Ltda, etc.) |
| codigo_organizacion_juridica | text | Código de organización jurídica |
| organizacion_juridica | text | Naturaleza jurídica (Persona Natural, Persona Jurídica, ESAL) |
| codigo_categoria_matricula | text | Código de la categoría de matrícula |
| categoria_matricula | text | Categoría de matrícula |
| codigo_estado_matricula | text | Código del estado de la matrícula |
| estado_matricula | text | Estado actual (Activa, Cancelada, etc.) |
| clase_identificacion_RL | text | Tipo de documento del representante legal |
| num_identificacion_representante_legal | text | Número de identificación del representante legal |
| representante_legal | text | Nombre del representante legal |
| fecha_actualizacion | text (timestamp) | Fecha y hora de la última sincronización del registro en RUES |

## Ficha técnica — Fuente 2: TRM (Tasa de Cambio Representativa del Mercado)

| Campo | Detalle |
|---|---|
| **Nombre del dataset** | Tasa de Cambio Representativa del Mercado - Histórico |
| **Origen** | Portal de Datos Abiertos de Colombia (datos.gov.co) — API REST Socrata (SoQL) |
| **Ficha del dataset (de donde se sacó la info)** | https://www.datos.gov.co/Econom-a-y-Finanzas/Tasa-de-Cambio-Representativa-del-Mercado-Historic/mcec-87by/about_data |
| **API (recurso)** | `https://www.datos.gov.co/resource/mcec-87by.json` |
| **Método de extracción** | HTTP GET con carga **incremental** (`$where` sobre `vigenciadesde`) |
| **Volumen exacto** | ~8.326 registros históricos totales en la fuente (serie diaria desde 1991); la carga inicial del pipeline se limita al **último año** (~250 registros) + 1 registro nuevo cada día hábil |
| **Tamaño aproximado** | 4 columnas, dataset pequeño en filas pero con recencia diaria |
| **Recencia** | Se actualiza diariamente — fuente ideal para demostrar carga incremental |
| **Carga usada en el pipeline** | Carga inicial del último año + **incremental** (`append`) en corridas posteriores, ver `03_Ingesta_Bronze_TRM.ipynb` |
| **Tabla Bronze** | `Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api` |

### Diccionario de datos inicial (TRM)

| Columna | Tipo original (Socrata) | Descripción |
|---|---|---|
| valor | number | Valor de la TRM (pesos colombianos por dólar) |
| unidad | text | Código de la divisa según ISO 4217 (ej. `COP`) |
| vigenciadesde | calendar_date | Fecha de inicio de vigencia de la tasa |
| vigenciahasta | calendar_date | Fecha de fin de vigencia de la tasa |

## Ficha técnica — Fuente 3: Catálogo CIIU (Actividades Económicas)

| Campo | Detalle |
|---|---|
| **Nombre del dataset** | Catálogo de actividades económicas |
| **Origen** | Portal de Datos Abiertos de Colombia (datos.gov.co) — API REST Socrata (SoQL) |
| **Ficha del dataset** | https://www.datos.gov.co/Econom-a-y-Finanzas/Cat-logo-de-actividades-econ-micas/nuke-fusu/about_data |
| **API (recurso)** | `https://www.datos.gov.co/resource/nuke-fusu.json` |
| **Método de extracción** | HTTP GET, carga completa (catálogo estático) |
| **Volumen exacto** | 499 registros |
| **Tamaño aproximado** | 5 columnas, dataset pequeño (tabla de referencia/lookup) |
| **Recencia** | No aplica — catálogo de códigos, no cambia con el tiempo |
| **Carga usada en el pipeline** | Completa (`overwrite`) en cada ejecución, ver `04_Ingesta_Bronze_CIIU.ipynb` |
| **Tabla Bronze** | `Datos_Empresas.bronze.DE_Semiestructurado_ActividadesEconomicas_Api` |

### Diccionario de datos inicial (Catálogo CIIU)

| Columna | Tipo original (Socrata) | Descripción |
|---|---|---|
| id | number | Identificador interno del catálogo |
| code | text | Código CIIU (mismo formato de 4 dígitos que `cod_ciiu_act_econ_pri` en RUES) |
| description | text | Descripción de la actividad económica |
| status | number | Estado del código en el catálogo |
| version | number | Versión de la clasificación CIIU |

## Ficha técnica — Fuente 4: SECOP II (Contratos Públicos)

| Campo | Detalle |
|---|---|
| **Nombre del dataset** | Contratos Públicos SECOP II |
| **Origen** | Portal de Datos Abiertos de Colombia (datos.gov.co) — API REST Socrata (SoQL) |
| **Ficha del dataset** | https://www.datos.gov.co/Orden-Publico-y-Seguridad/SECOP-II-2024-Cueros/jbjy-vk9h/about_data |
| **API (recurso)** | `https://www.datos.gov.co/resource/jbjy-vk9h.json` |
| **Método de extracción** | HTTP GET paginado (`$limit` / `$offset`), formato JSON, carga **incremental** |
| **Volumen exacto** | ~6.060.277 registros totales en la fuente (verificado con `$select=count(*)` el 2026-09-18) |
| **Tamaño aproximado** | 85 columnas de texto × ~6M filas → varios GB en crudo (JSON) |
| **Recencia** | Actualización continua; el pipeline filtra por `ultima_actualizacion` |
| **Carga usada en el pipeline** | Modo diario incremental (watermark sobre `ultima_actualizacion`) o mensual (`replaceWhere`), ver `06_Ingesta_Bronze_SECOP.ipynb` |
| **Tabla Bronze** | `Datos_Empresas.bronze.DE_Semiestructurado_SECOP_Contratos_Api` |

**Por qué esta fuente complementa al proyecto**: SECOP II trae la dimensión de contratación pública estatal — qué entidades contratan, con qué proveedores, por qué montos y bajo qué modalidad. El campo `documento_proveedor` es la llave de unión con RUES (`numero_identificacion`/`nit`), permitiendo en la Capa Silver cruzar el perfil empresarial del registro mercantil con su actividad contractual real en el Estado.

### Diccionario de datos inicial (SECOP II)

Todas las columnas llegan como `text` desde la API. Se listan las columnas de mayor relevancia para el análisis; el diccionario completo de 85 columnas está disponible en el output de `DESCRIBE EXTENDED` del notebook `06_Ingesta_Bronze_SECOP.ipynb`.

| Columna | Tipo original | Descripción |
|---|---|---|
| nombre_entidad | text | Nombre de la entidad contratante |
| nit_entidad | text | NIT de la entidad contratante |
| departamento | text | Departamento donde opera la entidad |
| ciudad | text | Ciudad donde opera la entidad |
| orden | text | Orden de la entidad (Nacional, Territorial) |
| sector | text | Sector al que pertenece la entidad |
| rama | text | Rama del poder público |
| entidad_centralizada | text | Centralizada o Descentralizada |
| proceso_de_compra | text | ID del proceso de compra en SECOP |
| id_contrato | text | ID único del contrato |
| referencia_del_contrato | text | Referencia interna del contrato |
| estado_contrato | text | Estado del contrato (Cerrado, Modificado, terminado, etc.) |
| codigo_de_categoria_principal | text | Código UNSPSC de la categoría del contrato |
| descripcion_del_proceso | text | Descripción del proceso de contratación |
| tipo_de_contrato | text | Tipo de contrato (Prestación de servicios, Compraventa, etc.) |
| modalidad_de_contratacion | text | Modalidad (Contratación directa, Licitación pública, etc.) |
| fecha_de_firma | text (ISO `YYYY-MM-DDTHH:MM:SS.fff`) | Fecha de firma del contrato |
| fecha_de_inicio_del_contrato | text (ISO) | Fecha de inicio de ejecución |
| fecha_de_fin_del_contrato | text (ISO) | Fecha de fin de ejecución |
| tipodocproveedor | text | Tipo de documento del proveedor (NIT, Cédula, etc.) |
| documento_proveedor | text | Número de identificación del proveedor — **llave de unión con RUES** |
| proveedor_adjudicado | text | Nombre del proveedor adjudicado |
| es_grupo | text | Si el proveedor es un grupo empresarial (Sí/No) |
| es_pyme | text | Si el proveedor es una PYME (Sí/No) |
| valor_del_contrato | text (numérico como texto) | Valor total del contrato en COP |
| valor_de_pago_adelantado | text (numérico) | Valor pagado por adelantado |
| valor_facturado | text (numérico) | Valor facturado hasta la fecha |
| valor_pendiente_de_pago | text (numérico) | Valor pendiente de pago |
| valor_pagado | text (numérico) | Valor total pagado |
| dias_adicionados | text (numérico) | Días adicionados al plazo original |
| ultima_actualizacion | text (ISO) | Fecha de última actualización del registro — **watermark del pipeline incremental** |
| objeto_del_contrato | text | Objeto/propósito del contrato |
| duraci_n_del_contrato | text | Duración del contrato (ej. "30 Dia(s)") |
| origen_de_los_recursos | text | Origen de los recursos (Recursos Propios, Distribuido, etc.) |
| destino_gasto | text | Destino del gasto (Funcionamiento, Inversión) |

**Columnas de auditoría** (agregadas por el pipeline, no provienen de la API):

| Columna | Tipo | Descripción |
|---|---|---|
| _ingested_at | timestamp | Fecha y hora de ingesta en Bronze |
| _source | string | URL de la API de origen |

**Nota**: El campo `urlproceso` llega como objeto JSON anidado desde la API y se serializa a string JSON para persistirlo en Delta Lake, sin alterar su contenido (principio Bronze).

---

# Sección 2: Diagrama de Arquitectura Inicial

El diagrama ilustra: las cuatro fuentes de origen, el mecanismo de ingesta de cada una (batch mensual, incremental o carga completa), la Capa Bronze en Unity Catalog (catálogo `Datos_Empresas`, esquema `bronze`) y los puntos donde se harán los `JOIN` entre RUES y el catálogo CIIU, y entre RUES y SECOP, en la futura Capa Silver.

### Diagrama de Arquitectura

![Diagrama de arquitectura](./Diagrama%20de%20Arquitectura%20Inicial%20Imagen.png)

[Ver imagen en Databricks](https://dbc-7478df09-02e1.cloud.databricks.com/editor/files/3744392461485816?o=7474658561793168)

**Notas de arquitectura**:
- Las cuatro fuentes se consultan directamente vía HTTP desde Databricks (no requieren S3 ni Auto Loader, dado que el origen ya es una API REST).
- Las tablas Bronze son Delta Lake puro, dentro del catálogo `Datos_Empresas`, esquema `bronze`.
- RUES se carga por mes o en modo diario incremental; TRM es incremental (`append`) día a día; el catálogo CIIU es una tabla de referencia estática que se recarga completa (`overwrite`); SECOP se carga en modo diario incremental (watermark sobre `ultima_actualizacion`).

---

## Estructura de notebooks del proyecto

| Notebook | Sección | Contenido |
|---|---|---|
| `00_Informe.ipynb` (este notebook) | 1 y 2 | Contexto de negocio, ficha técnica y diccionario de datos de las 4 fuentes, diagrama de arquitectura |
| [`01_Configuracion_Inicial.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/3744392461485810?o=7474658561793168) | — | Creación del catálogo `Datos_Empresas` y el esquema `bronze` en Unity Catalog |
| [`02_Ingesta_Bronze_RUES.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/3744392461485811?o=7474658561793168) | 3 (Fuente 1) | Pipeline PySpark de RUES (modo mensual o diario incremental) |
| [`03_Ingesta_Bronze_TRM.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/3744392461485812?o=7474658561793168) | 3 (Fuente 2) | Pipeline PySpark de TRM con carga **incremental obligatoria** |
| [`04_Ingesta_Bronze_CIIU.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/3744392461485813?o=7474658561793168) | 3 (Fuente 3) | Pipeline PySpark del catálogo CIIU código→descripción |
| [`06_Ingesta_Bronze_SECOP.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/2279170452559878?o=7474658561793168) | 3 (Fuente 4) | Pipeline PySpark de SECOP II con carga incremental |
| [`05_Validaciones_Bronze.ipynb`](https://dbc-7478df09-02e1.cloud.databricks.com/editor/notebooks/1875149303955997?o=7474658561793168) | 4 | Nulos, duplicados, outliers e integridad referencial (RUES↔CIIU, SECOP↔RUES) |

Orden de ejecución recomendado: `00` → `01` → `02` → `03` → `04` → `06` → `05`.